# Notebook 4 — Date & Time Feature Engineering
Using `signup_date` from `telecom_customers.csv` and `transaction_date` from
`telecom_transactions.csv`.

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
transactions = pd.read_csv("./telecom_transactions.csv", parse_dates=["transaction_date"])
customers[["customer_id","signup_date"]].head()

## 1. Calendar Component Extraction

**When to use:** whenever a raw timestamp might hide a cyclical or seasonal business
pattern — e.g. sign-ups spiking at certain months (promotions), or purchases
clustering on weekends.

In [ ]:
customers["signup_year"] = customers["signup_date"].dt.year
customers["signup_month"] = customers["signup_date"].dt.month
customers["signup_day"] = customers["signup_date"].dt.day
customers["signup_dayofweek"] = customers["signup_date"].dt.dayofweek   # 0=Mon
customers["signup_week"] = customers["signup_date"].dt.isocalendar().week
customers["signup_quarter"] = customers["signup_date"].dt.quarter
customers["is_weekend_signup"] = customers["signup_dayofweek"].isin([5,6]).astype(int)
customers["is_month_start"] = customers["signup_date"].dt.is_month_start.astype(int)
customers["is_month_end"] = customers["signup_date"].dt.is_month_end.astype(int)

customers[["signup_date","signup_year","signup_month","signup_dayofweek",
           "signup_quarter","is_weekend_signup"]].head()

## 2. Days Since Event / Customer Tenure

This is the single highest-value date feature in almost any subscription-business
dataset. It converts an absolute timestamp (useless for generalization — every
customer has a nearly unique signup date) into a **relative, comparable duration**.

In [ ]:
reference_date = pd.Timestamp("2024-06-30")  # the "as of" date the dataset was built for
customers["tenure_days"] = (reference_date - customers["signup_date"]).dt.days
customers["tenure_years"] = (customers["tenure_days"] / 365.25).round(2)

customers[["signup_date","tenure_days","tenure_years","tenure_months"]].head()

> **Production note:** `reference_date` must be the date the *prediction is being
> made*, not `today()` re-computed at training time vs a different value at serving
> time — inconsistency here is a subtle but common source of train/serve skew.

## 3. Recency & Frequency Features (from transaction log)

**When to use:** any time you have an event log per entity (purchases, logins, support
calls) — recency and frequency are the backbone of RFM analysis, widely used in
marketing, retention, and fraud models.

In [ ]:
last_tx = transactions.groupby("customer_id")["transaction_date"].max().rename("last_transaction_date")
tx_count = transactions.groupby("customer_id").size().rename("transaction_count")

rf = pd.concat([last_tx, tx_count], axis=1).reset_index()
rf["days_since_last_transaction"] = (reference_date - rf["last_transaction_date"]).dt.days

customers = customers.merge(rf, on="customer_id", how="left")
customers["transaction_count"] = customers["transaction_count"].fillna(0)
customers["days_since_last_transaction"] = customers["days_since_last_transaction"].fillna(
    customers["tenure_days"]  # customers with zero transactions: use full tenure as recency
)

customers[["customer_id","transaction_count","days_since_last_transaction"]].head()

**Business meaning:** `days_since_last_transaction` (Recency) is one of the strongest
churn predictors in subscription businesses — a customer who hasn't engaged recently
is at much higher risk, independent of how long they've been a customer overall.

## 4. Time Difference Between Consecutive Events

**When to use:** measuring engagement *rhythm* — e.g. average days between purchases,
which reveals whether a customer's behavior is accelerating or slowing down.

In [ ]:
tx_sorted = transactions.sort_values(["customer_id","transaction_date"])
tx_sorted["days_since_prev_tx"] = (
    tx_sorted.groupby("customer_id")["transaction_date"].diff().dt.days
)
avg_gap = tx_sorted.groupby("customer_id")["days_since_prev_tx"].mean().rename("avg_days_between_tx")

customers = customers.merge(avg_gap, on="customer_id", how="left")
customers[["customer_id","avg_days_between_tx"]].dropna().head()

## 5. Seasonality Indicator

**When to use:** businesses with known seasonal cycles (e.g. telecom promotions
cluster around New Year and back-to-school season).

In [ ]:
customers["signup_season"] = customers["signup_month"].map({
    12:"Winter",1:"Winter",2:"Winter",
    3:"Spring",4:"Spring",5:"Spring",
    6:"Summer",7:"Summer",8:"Summer",
    9:"Fall",10:"Fall",11:"Fall"
})
customers["signup_season"].value_counts()

## Summary — Feature Justification

| Feature | Source | Logic | Business Meaning | Leakage Risk | Decision |
|---|---|---|---|---|---|
| `tenure_days` / `tenure_years` | signup_date, reference_date | date subtraction | how established the relationship is | None if `reference_date` ≤ prediction time | **Retain** |
| `days_since_last_transaction` | transaction_date (max), reference_date | recency | disengagement = churn risk | None | **Retain** — strongest date-based feature here |
| `avg_days_between_tx` | transaction_date (diffs) | mean inter-event gap | engagement rhythm | None | **Retain** |
| `is_weekend_signup` | signup_date | day-of-week flag | acquisition channel proxy | None | **Retain**, low individual importance expected |
| `signup_season` | signup_month | calendar bucket | promo-cycle exposure | None | **Needs further analysis** — check with Feature Selection (Notebook 9) |